In [ ]:
!pip install ultralytics -q
print("Da cai dat xong ultralytics")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 6.8 MB/s eta 0:00:00
Da cai dat xong ultralytics


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

SEED = 1407

# ===== MODEL =====
BASELINE_COCO = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/YOLOv8n/baseline_100epochs_es/weights/best.pt"
# Model B cuoi cung sinh tu FILE 2 (COCO->CanTho->HCMC seed 1407)
MODEL_B_FINAL = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/finetune_hcmc_from_cantho_seed1407/weights/best.pt"

# ===== DATASET (goc tren Drive) =====
DRIVE_DIR_HCMC = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515"
DRIVE_DIR_COCO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515"
DATA_YAML_CANTHO = "/content/drive/MyDrive/Do an AI giao thong/Dataset/CanTho_701515/data.yaml"

# ===== DATASET (copy ve local) =====
LOCAL_DIR_HCMC = "/content/hcmc_701515"
LOCAL_DIR_COCO = "/content/coco_701515"

# ===== THU MUC LUU KET QUA =====
PROJECT_DIR = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline"
NAME_A = "finetune_hcmc_from_coco_seed1407"   # model A (seed moi)

# ===== CAU HINH HUAN LUYEN (dung Bang 3.4) =====
IMGSZ = 640
BATCH = 16
EPOCHS = 100
PATIENCE = 20
LR0 = 0.001

print("Kiem tra su ton tai:")
print("  Baseline COCO         :", os.path.exists(BASELINE_COCO))
print("  Model B cuoi (tu FILE 2):", os.path.exists(MODEL_B_FINAL))
print("  Dataset HCMC (Drive)  :", os.path.exists(DRIVE_DIR_HCMC))
print("  Dataset COCO (Drive)  :", os.path.exists(DRIVE_DIR_COCO))
print("  Yaml CanTho           :", os.path.exists(DATA_YAML_CANTHO))
if not os.path.exists(MODEL_B_FINAL):
    print("\n[LOI] Chua tim thay model B cuoi cung. Hay chay xong FILE 2 truoc!")

Kiem tra su ton tai:
  Baseline COCO         : True
  Model B cuoi (tu FILE 2): True
  Dataset HCMC (Drive)  : True
  Dataset COCO (Drive)  : True
  Yaml CanTho           : True


In [ ]:
import shutil, time

def copy_ds(drive_dir, local_dir, ten):
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
    t0 = time.time()
    try:
        shutil.copytree(drive_dir, local_dir)
        print(f"[OK] Da copy {ten} ve {local_dir} ({time.time()-t0:.1f}s)")
    except shutil.Error as e:
        print(f"[CANH BAO] {len(e.args[0])} file loi tam thoi khi copy {ten}.")

copy_ds(DRIVE_DIR_HCMC, LOCAL_DIR_HCMC, "HCMC")
copy_ds(DRIVE_DIR_COCO, LOCAL_DIR_COCO, "Vehicles-COCO")

DATA_YAML_HCMC_LOCAL = f"{LOCAL_DIR_HCMC}/data.yaml"
DATA_YAML_COCO_LOCAL = f"{LOCAL_DIR_COCO}/data.yaml"
print()
print("Yaml HCMC (local):", os.path.exists(DATA_YAML_HCMC_LOCAL))
print("Yaml COCO (local):", os.path.exists(DATA_YAML_COCO_LOCAL))

[OK] Da copy HCMC ve /content/hcmc_701515 (789.5s)
[OK] Da copy Vehicles-COCO ve /content/coco_701515 (996.0s)

Yaml HCMC (local): True
Yaml COCO (local): True


In [5]:
from ultralytics import YOLO

model_A = YOLO(BASELINE_COCO)

results_A = model_A.train(
    data=DATA_YAML_HCMC_LOCAL,
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    lr0=LR0, seed=SEED,
    project=PROJECT_DIR, name=NAME_A, exist_ok=True, verbose=True,
)
print("\nDa huan luyen xong luong A (COCO -> HCMC, seed 1407).")
BEST_A = os.path.join(PROJECT_DIR, NAME_A, "weights", "best.pt")
print("Model A luu tai:", BEST_A)
print("Ton tai:", os.path.exists(BEST_A))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hcmc_701515/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscri

In [6]:
BEST_A = os.path.join(PROJECT_DIR, NAME_A, "weights", "best.pt")
model_A_eval = YOLO(BEST_A)

# --- A tren test HCMC (in-domain) ---
mA_hcmc = model_A_eval.val(data=DATA_YAML_HCMC_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_A + "_eval_hcmc", exist_ok=True)
A_hcmc_map50, A_hcmc_map = float(mA_hcmc.box.map50), float(mA_hcmc.box.map)

# --- A tren test COCO (forgetting) ---
mA_coco = model_A_eval.val(data=DATA_YAML_COCO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_A + "_eval_coco", exist_ok=True)
A_coco_map50, A_coco_map = float(mA_coco.box.map50), float(mA_coco.box.map)

# --- A tren test CanTho ---
mA_cantho = model_A_eval.val(data=DATA_YAML_CANTHO, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name=NAME_A + "_eval_cantho", exist_ok=True)
A_cantho_map50, A_cantho_map = float(mA_cantho.box.map50), float(mA_cantho.box.map)

print("===== MODEL A (COCO->HCMC seed 1407) =====")
print(f"Test HCMC   : mAP@0.5={A_hcmc_map50:.4f}  mAP@0.5:0.95={A_hcmc_map:.4f}")
print(f"Test COCO   : mAP@0.5={A_coco_map50:.4f}  mAP@0.5:0.95={A_coco_map:.4f}")
print(f"Test CanTho : mAP@0.5={A_cantho_map50:.4f}  mAP@0.5:0.95={A_cantho_map:.4f}")

Ultralytics 8.4.144 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.6±0.1 ms, read: 40.6±17.9 MB/s, size: 55.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/labels.cache... 2267 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2267/2267 594.3Mit/s 0.0s
val: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/images/cam_01_000596_jpg.rf.e27909b3bca621ade420095897f7c5e7.jpg: 1 duplicate labels removed
val: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicle Counting AIC HCMC 2020.v1i.yolov8_701515/test/images/cam_01_001195_jpg.rf.99126ca4b71bfad3e186294ec9f1f02f.jpg: 1 d

In [ ]:
model_B_eval = YOLO(MODEL_B_FINAL)

mB_hcmc = model_B_eval.val(data=DATA_YAML_HCMC_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name="B_seed1407_eval_hcmc", exist_ok=True)
B_hcmc_map50, B_hcmc_map = float(mB_hcmc.box.map50), float(mB_hcmc.box.map)

mB_coco = model_B_eval.val(data=DATA_YAML_COCO_LOCAL, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name="B_seed1407_eval_coco", exist_ok=True)
B_coco_map50, B_coco_map = float(mB_coco.box.map50), float(mB_coco.box.map)

mB_cantho = model_B_eval.val(data=DATA_YAML_CANTHO, split="test", imgsz=IMGSZ, batch=BATCH,
    project=PROJECT_DIR, name="B_seed1407_eval_cantho", exist_ok=True)
B_cantho_map50, B_cantho_map = float(mB_cantho.box.map50), float(mB_cantho.box.map)

print("===== MODEL B (COCO->CanTho->HCMC seed 1407) =====")
print(f"Test HCMC   : mAP@0.5={B_hcmc_map50:.4f}  mAP@0.5:0.95={B_hcmc_map:.4f}")
print(f"Test COCO   : mAP@0.5={B_coco_map50:.4f}  mAP@0.5:0.95={B_coco_map:.4f}")
print(f"Test CanTho : mAP@0.5={B_cantho_map50:.4f}  mAP@0.5:0.95={B_cantho_map:.4f}")

In [8]:
# ===== Ket qua seed 42 da co truoc do (de doi chieu) =====
A42 = {"hcmc":0.9539, "coco":0.0931, "cantho":0.7635}
B42 = {"hcmc":0.9530, "coco":0.0843, "cantho":0.7500}

print("="*82)
print("BANG SO SANH ABLATION - SEED 1407  (mAP@0.5)")
print("="*82)
print(f"{'Phep danh gia':28s} {'A (COCO->HCMC)':>16s} {'B (COCO->CT->HCMC)':>20s} {'Chenh (A-B)':>12s}")
print("-"*82)
for ten, a, b in [
    ("Test HCMC (in-domain)", A_hcmc_map50, B_hcmc_map50),
    ("Test COCO (forgetting)", A_coco_map50, B_coco_map50),
    ("Test CanTho", A_cantho_map50, B_cantho_map50),
]:
    print(f"{ten:28s} {a:>16.4f} {b:>20.4f} {a-b:>+12.4f}")
print("="*82)

print()
print("="*82)
print("DOI CHIEU 2 SEED (mAP@0.5 tren test HCMC - chi tieu quyet dinh)")
print("="*82)
print(f"{'':20s} {'Seed 42':>12s} {'Seed 1407':>12s}")
print(f"{'A (COCO->HCMC)':20s} {A42['hcmc']:>12.4f} {A_hcmc_map50:>12.4f}")
print(f"{'B (COCO->CT->HCMC)':20s} {B42['hcmc']:>12.4f} {B_hcmc_map50:>12.4f}")
print(f"{'Chenh lech (A-B)':20s} {A42['hcmc']-B42['hcmc']:>+12.4f} {A_hcmc_map50-B_hcmc_map50:>+12.4f}")
print("="*82)

print()
print("--- KET LUAN TU DONG ---")
d1407 = A_hcmc_map50 - B_hcmc_map50
d42 = A42['hcmc'] - B42['hcmc']
if abs(d1407) < 0.01:
    print(f"Voi seed 1407, chenh lech A-B tren test HCMC = {d1407:+.4f} (< 0.01).")
    print("=> Xac nhan lai: hai lo trinh cho hieu nang TUONG DUONG tren mien dich HCMC,")
    print("   nhat quan voi ket qua seed 42 (chenh {:+.4f}).".format(d42))
    print("   Ket luan 'buoc trung gian CanTho khong cai thien dang ke' duoc cung co")
    print("   qua 2 lan chay voi 2 seed khac nhau.")
else:
    print(f"Voi seed 1407, chenh lech A-B = {d1407:+.4f} (>= 0.01), lon hon so voi seed 42.")
    print("=> Can xem xet ky: xu huong co the khong hoan toan on dinh giua cac seed,")
    print("   nen trinh bay ca 2 ket qua mot cach minh bach trong bao cao.")

BANG SO SANH ABLATION - SEED 1407  (mAP@0.5)
Phep danh gia                  A (COCO->HCMC)   B (COCO->CT->HCMC)  Chenh (A-B)
----------------------------------------------------------------------------------
Test HCMC (in-domain)                  0.9550               0.9545      +0.0005
Test COCO (forgetting)                 0.0959               0.0659      +0.0299
Test CanTho                            0.7757               0.7779      -0.0022

DOI CHIEU 2 SEED (mAP@0.5 tren test HCMC - chi tieu quyet dinh)
                          Seed 42    Seed 1407
A (COCO->HCMC)             0.9539       0.9550
B (COCO->CT->HCMC)         0.9530       0.9545
Chenh lech (A-B)          +0.0009      +0.0005

--- KET LUAN TU DONG ---
Voi seed 1407, chenh lech A-B tren test HCMC = +0.0005 (< 0.01).
=> Xac nhan lai: hai lo trinh cho hieu nang TUONG DUONG tren mien dich HCMC,
   nhat quan voi ket qua seed 42 (chenh +0.0009).
   Ket luan 'buoc trung gian CanTho khong cai thien dang ke' duoc cung co
   qua 